# Aprendizado de Máquina — Lista prática 06

## Árvores de Regressão e *Ensembles*

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A Lista Teórica 06 deduziu que a variância de um *ensemble* trava num piso
$\rho\,v(x)$, e que reduzir $\rho$ vale mais do que aumentar $B$. Esta lista
**mede as três quantidades** — $v$, $\rho$ e o piso — e verifica se a floresta
aleatória faz o que promete.

> **a floresta não é ``bagging com mais aleatoriedade''. Ela paga viés para
> comprar correlação, e dá para ver o preço e a compra na mesma tabela.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — uma árvore, e a profundidade

Comece pelo tijolo. Carregue o `superconductivity.csv`, fique com 3 000
observações de treino, e ajuste árvores de várias profundidades.

In [ ]:
_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
X_todos = df.drop(columns="critical_temp").values
y_todos = df["critical_temp"].values

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    X_todos, y_todos, train_size=3000, test_size=4000, random_state=2026)

print("treino", X_tr.shape, " teste", X_te.shape)

In [ ]:
print("  profundidade   folhas   R2 treino   R2 teste")
for prof in (2, 5, 10, None):
    arvore = DecisionTreeRegressor(max_depth=..., random_state=0).fit(X_tr, y_tr)   # (a)
    print(f"  {str(prof):>12s}   {arvore.get_n_leaves():6d}   "
          f"{arvore.score(X_tr, y_tr):9.4f}   {...:8.4f}")      # (b)

---
## Exercício 2 — medindo $v$, $\rho$ e o piso

Aqui está o exercício central da lista. A fórmula da Lista Teórica 06 diz

$$\operatorname{Var}(\bar g(x)) \;=\; \rho\,v(x) \;+\; \frac{1-\rho}{B}\,v(x),$$

e queremos as três quantidades **medidas**.

Há uma sutileza que decide o experimento: $\rho$ é a correlação entre duas
árvores **sobre a aleatoriedade que as gerou** — o que inclui o sorteio do
conjunto de treino. Não dá para estimá-la de uma única floresta. É preciso
ajustar $R$ florestas em $R$ conjuntos de treino **independentes**:

- $v(x)$: variância entre as $B$ árvores, dentro de cada floresta;
- $\operatorname{Var}(\bar g(x))$: variância da predição da floresta, entre as $R$ florestas;
- $\rho$: isolado da fórmula, $\rho = \dfrac{\operatorname{Var}(\bar g) - v/B}{v - v/B}$.

Usamos a população sintética da figura da nota ($n=150$, $p=8$, ruído $1{,}5$),
porque ali dá para sortear conjuntos de treino à vontade.

In [ ]:
def alvo(M):
    return np.sin(1.5 * M[:, 0]) + 0.8 * M[:, 1] * M[:, 2] + 0.5 * M[:, 0] ** 2


n, d, ruido = 150, 8, 1.5
B, R = 100, 40

rng = np.random.default_rng(2026)
X0 = rng.uniform(-2, 2, size=(400, d))       # pontos onde medimos

print("  max_features       v(x)    Var(gbar)      rho     piso rho*v   EQM teste")
for mf, rotulo in [(None, "d (bagging)"), (1 / 3, "d/3 (floresta)"), (0.15, "0,15 d")]:
    rng = np.random.default_rng(2026)
    gbar = np.zeros((R, len(X0)))
    vs, eqm = [], []

    for r_ in range(R):
        X = rng.uniform(-2, 2, size=(n, d))
        y = alvo(X) + rng.normal(0, ruido, size=n)
        floresta = RandomForestRegressor(n_estimators=B, max_features=...,        # (a)
                                         random_state=r_, n_jobs=-1).fit(X, y)
        # predicao de CADA arvore, separadamente
        P = np.column_stack([arv.predict(X0) for arv in ...])   # (b)
        vs.append(P.var(axis=1, ddof=1).mean())        # variancia ENTRE arvores
        gbar[r_] = ...                    # (c) a predicao do ensemble
        eqm.append(np.mean((floresta.predict(X0) - alvo(X0)) ** 2))

    v = np.mean(vs)
    var_gbar = gbar.var(axis=0, ddof=1).mean()         # variancia ENTRE florestas
    rho = ...           # (d) isole rho da formula

    print(f"  {rotulo:16s} {v:7.3f}   {var_gbar:8.3f}   {rho:8.4f}   {rho * v:8.3f}"
          f"   {np.mean(eqm):8.4f}")

> **Sua vez.** Repita a linha da floresta com $B=10$ em vez de $B=100$. O $\rho$
> medido muda? E a $\operatorname{Var}(\bar g)$? Compare com o que a fórmula
> prevê.

---
## Exercício 3 — $B$ grande: inofensivo ou perigoso?

Na mesma população, compare o que acontece quando $B$ cresce nos dois
*ensembles*. Guarde também o **erro de treino** — é ele que denuncia o mecanismo.

In [ ]:
rng = np.random.default_rng(12)
X = rng.uniform(-2, 2, size=(n, d))
y = alvo(X) + rng.normal(0, ruido, size=n)
X_gb_te = rng.uniform(-2, 2, size=(4000, d))
y_gb_te = alvo(X_gb_te) + rng.normal(0, ruido, size=4000)

print("floresta (max_features=1/3):")
for Bf in (1, 10, 100, 800):
    m = RandomForestRegressor(n_estimators=Bf, max_features=1/3, random_state=0).fit(X, y)
    print(f"   B={Bf:4d}: teste {np.mean((y_gb_te - m.predict(X_gb_te)) ** 2):.4f}"
          f"   treino {np.mean((y - m.predict(X)) ** 2):.4f}")

In [ ]:
gb = GradientBoostingRegressor(n_estimators=800, learning_rate=0.05,
                               max_depth=3, random_state=0).fit(X, y)

# staged_predict devolve a predicao do ensemble apos CADA arvore acrescentada
erro_te = np.array([np.mean((y_gb_te - p) ** 2) for p in ...])   # (a)
erro_tr = np.array([np.mean((y - p) ** 2) for p in gb.staged_predict(X)])

melhor_B = ...                     # (b) +1 porque B comeca em 1
print(f"boosting: minimo em B={melhor_B} (EQM {erro_te.min():.4f}); "
      f"B=800 -> {erro_te[-1]:.4f} ({100 * (erro_te[-1] / erro_te.min() - 1):+.1f}%)")
for b in (1, 10, 32, 100, 300, 800):
    print(f"   B={b:4d}: teste {erro_te[b-1]:.4f}   treino {erro_tr[b-1]:.4f}")

---
## Exercício 4 — "*boosting* superajusta com $B$ grande" é sempre verdade?

A frase do Exercício 3 é a que se costuma repetir. Teste-a **no outro conjunto de
dados**: o mesmo *boosting*, mesmos hiperparâmetros de família, agora nas 3 000
observações do `superconductivity` do Exercício 1.

In [ ]:
gb2 = GradientBoostingRegressor(n_estimators=800, learning_rate=0.1,
                                max_depth=3, random_state=0).fit(X_tr, y_tr)
erro2 = np.array([np.mean((y_te - p) ** 2) for p in gb2.staged_predict(X_te)])

print(f"minimo em B={...} (EQM {erro2.min():.2f})")   # (a)
print(f"em B=800 -> {erro2[-1]:.2f} "
      f"({100 * (...):+.1f}% acima do minimo)")     # (b)

A parada antecipada faz isso sozinha: separa uma fração dos dados de treino,
acompanha o erro nela a cada iteração e para quando ele deixa de melhorar. Teste
na população **pequena**, que é onde há o que evitar.

In [ ]:
gb_parada = GradientBoostingRegressor(
    n_estimators=800, learning_rate=0.05, max_depth=3,
    n_iter_no_change=..., validation_fraction=0.2, random_state=0).fit(X, y)   # (a)

print(f"parou em n_estimators_ = {gb_parada.n_estimators_}")
print(f"EQM de teste: {np.mean((y_gb_te - gb_parada.predict(X_gb_te)) ** 2):.4f}")
print(f"(o minimo verdadeiro era {erro_te.min():.4f} em B={melhor_B};"
      f" sem parar, B=800 dava {erro_te[-1]:.4f})")

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | a árvore sem limite usa 2 594 folhas para 3 000 pontos: $R^2$ de treino 0,9922 e de teste 0,7721 |
| 2 | a floresta **corta $\rho$ pela metade** (0,147 → 0,0735) e paga com $v$ maior (3,28 → 3,80) |
| 2 | o piso $\rho v$ já é 88% da variância com $B=100$ — acrescentar árvores não tem para onde ir |
| 2 | reduzir `max_features` além de $p/3$ continua baixando $\rho$ e **piora** o EQM |
| 3 | o *boosting* leva o erro de treino a 0,0011 (interpola); a floresta estaciona em 0,46 |
| 4 | a curva em U do *boosting* **some** com $n=3000$: em $B=800$ o erro ainda está caindo |
| 4 | a parada antecipada evita metade do estrago, mas para em 154 quando o ótimo era 32 |

**A seguir.** A Aula 07 arruma a casa: `Pipeline`, `ColumnTransformer` e a
disciplina que impede que a padronização, a imputação ou a seleção de variáveis
vejam a dobra de validação.